# APDL Preview for `bcc`

Generate APDL command text for the `bcc` unit cell and inspect it before pasting it into MAPDL/APDL.

In [38]:
%pip install -e .

from pathlib import Path
import sys

%load_ext autoreload
%autoreload 2

project_root = Path.cwd().resolve()

src = project_root / "src"

for p in [str(src), str(project_root)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("project_root:", project_root)
print("src:", src)
print("sys.path[0:3]:", sys.path[:3])

Obtaining file:///C:/Users/USER/Documents/parametric_lattice
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for parametric-lattice (pyproject.toml): started
  Building editable for parametric-lattice (pyproject.toml): finished with status 'done'
  Created wheel for parametric-lattice: filename=parametric_lattice-0.1.0-0.editable-py3-none-any.whl size=4660 sha256=41d47924cbf3717e2408db89834cff55a0c9151589d4b475a35a0c27319fe74e
  Stored in directory: C:\Users\USER\AppData\Local\Temp\pip-ephem-wheel-cache-le05jhox\wheels\db\36\3

In [39]:
from apdl_preview import build_sim_case


sim_case = build_sim_case(
    "bcc",
    element_type="SOLID187",
    sim_type="xx",  # static compression along x
    size_xyz=(10.0, 10.0, 10.0),
    radius=1,
    e_mod=210000.0,
    nu=0.3,
    density=7.85e-9,
    max_element_size=0.4,
    strain=0.01,
    n_substeps=1,
    kappa=0.85
)

print(sim_case.to_string())

SOLID187__Radius:1.0000000000000000e+00__Cell:bcc__Size:1.0000000000000000e+01,1.0000000000000000e+01,1.0000000000000000e+01__MaxElementSize:4.0000000000000002e-01__E:2.1000000000000000e+05__Nu:2.9999999999999999e-01__Density:7.8500000000000008e-09__SimType:xx__Strain:1.0000000000000000e-02__Substeps:1


In [40]:
from custom_io.mapdl.apdl_io import generate_apdl_script
from pipeline import build_pipeline
from post.pipeline import post_commands

apdl_cmd = generate_apdl_script(
    build_pipeline(
        sim_case,     
        save_intermediate = False,)
    + ("/POST1",)
    # + post_commands(
    #     sim_case=sim_case,
    #     needed={"boundary_traction": 9},
    # )
)
print(apdl_cmd)

/CLEAR,START
/UNITS,MPA
/PREP7
! Define solid element type
ET,1,187
! ============================================================
! GEOMETRY DEFINITION
! ============================================================
! --- Create keypoints
K,1,-5,-5,-5                    ! 0: n 0.0 0.0 0.0
K,2,5,-5,-5                     ! 1: n 1.0 0.0 0.0
K,3,-5,5,-5                     ! 2: n 0.0 1.0 0.0
K,4,5,5,-5                      ! 3: n 1.0 1.0 0.0
K,5,0,0,0                       ! 4: n 0.5 0.5 0.5
K,6,-5,-5,5                     ! 5: n 0.0 0.0 1.0
K,7,5,-5,5                      ! 6: n 1.0 0.0 1.0
K,8,-5,5,5                      ! 7: n 0.0 1.0 1.0
K,9,5,5,5                       ! 8: n 1.0 1.0 1.0
! --- Create beam orientation keypoints from edge normal vectors
K,10,-5,-5,-4                   !  0: e_n [0. 0. 1.]
K,11,-5,-4,-5                   !  0: e_b [0. 1. 0.]
K,12,-5,-5,-6                   !  1: e_n [ 0.  0. -1.]
K,13,-4,-5,-5                   !  1: e_b [ 1. -0.  0.]
K,14,-5,-4.38762756

In [41]:
from custom_io.excel_io import run_cases


# run_cases((sim_case,))

In [42]:
# 6597.34473
# 38819.21

# 19819.92
# -1649.336
